In [1]:
# Cell 1 - PDF imports
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, HRFlowable
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from pathlib import Path
from datetime import datetime
from PIL import Image as PILImage
import io
import os

BASE_DIR = Path("..").resolve()
print("✅ PDF libraries imported!")

✅ PDF libraries imported!


In [2]:
# Cell 2 - PDF Generator
def generate_pdf_report(patient_data, prediction_result, 
                         image_path, output_path):
    
    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=A4,
        rightMargin=50,
        leftMargin=50,
        topMargin=50,
        bottomMargin=50
    )
    
    styles = getSampleStyleSheet()
    story  = []
    
    # ── HEADER ──
    title_style = ParagraphStyle(
        'Title',
        parent=styles['Title'],
        fontSize=24,
        textColor=colors.HexColor('#1a1a2e'),
        spaceAfter=5,
        alignment=TA_CENTER
    )
    subtitle_style = ParagraphStyle(
        'Subtitle',
        parent=styles['Normal'],
        fontSize=11,
        textColor=colors.HexColor('#555555'),
        spaceAfter=20,
        alignment=TA_CENTER
    )
    heading_style = ParagraphStyle(
        'Heading',
        parent=styles['Heading2'],
        fontSize=13,
        textColor=colors.HexColor('#4361ee'),
        spaceBefore=15,
        spaceAfter=8
    )
    normal_style = ParagraphStyle(
        'Normal',
        parent=styles['Normal'],
        fontSize=10,
        spaceAfter=5
    )
    
    # Title
    story.append(Paragraph("🩺 Skin Disease Prediction Report", title_style))
    story.append(Paragraph("AI-Powered Skin Analysis | For Educational Purposes Only", subtitle_style))
    story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor('#4361ee')))
    story.append(Spacer(1, 15))
    
    # ── PATIENT INFO ──
    story.append(Paragraph("👤 Patient Information", heading_style))
    
    patient_table_data = [
        ["Field", "Details"],
        ["Name",     patient_data.get("name", "N/A")],
        ["Age",      str(patient_data.get("age", "N/A"))],
        ["Gender",   patient_data.get("gender", "N/A")],
        ["Species",  patient_data.get("species", "Human")],
        ["Duration", patient_data.get("duration", "N/A")],
        ["Itchy",    "Yes" if patient_data.get("itchy") else "No"],
        ["Spreading","Yes" if patient_data.get("spreading") else "No"],
        ["Allergies",patient_data.get("allergies", "None")],
        ["Date",     datetime.now().strftime("%d-%m-%Y %H:%M")]
    ]
    
    patient_table = Table(patient_table_data, 
                          colWidths=[2*inch, 4*inch])
    patient_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#4361ee')),
        ('TEXTCOLOR',  (0, 0), (-1, 0), colors.white),
        ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE',   (0, 0), (-1, 0), 11),
        ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor('#f8f9fa')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), 
         [colors.white, colors.HexColor('#f0f4ff')]),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.HexColor('#e0e0e0')),
        ('FONTSIZE',   (0, 1), (-1, -1), 10),
        ('PADDING',    (0, 0), (-1, -1), 8),
    ]))
    
    story.append(patient_table)
    story.append(Spacer(1, 15))
    
    # ── PREDICTION RESULTS ──
    story.append(Paragraph("📊 Prediction Results", heading_style))
    
    severity = prediction_result.get("severity", "N/A")
    severity_colors_map = {
        "Mild":     "#2dc653",
        "Moderate": "#f4a261",
        "Severe":   "#e63946"
    }
    severity_color = severity_colors_map.get(severity, "#4361ee")
    
    results_data = [
        ["Metric", "Result"],
        ["Detected Disease", prediction_result.get("disease", "N/A")],
        ["Confidence",       f"{prediction_result.get('confidence', 0)}%"],
        ["Severity Level",   severity],
    ]
    
    results_table = Table(results_data, colWidths=[2*inch, 4*inch])
    results_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#4361ee')),
        ('TEXTCOLOR',  (0, 0), (-1, 0), colors.white),
        ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTNAME',   (0, 1), (-1, -1), 'Helvetica-Bold'),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.HexColor('#e0e0e0')),
        ('FONTSIZE',   (0, 0), (-1, -1), 10),
        ('PADDING',    (0, 0), (-1, -1), 8),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1),
         [colors.white, colors.HexColor('#f0f4ff')]),
    ]))
    story.append(results_table)
    story.append(Spacer(1, 15))
    
    # ── TOP 5 PREDICTIONS ──
    story.append(Paragraph("🏆 Top 5 Predictions", heading_style))
    
    top5_data = [["Disease", "Confidence %"]]
    for disease, prob in prediction_result.get("top5", []):
        top5_data.append([
            disease.replace("_", " ").title(),
            f"{prob:.1f}%"
        ])
    
    top5_table = Table(top5_data, colWidths=[4*inch, 2*inch])
    top5_table.setStyle(TableStyle([
        ('BACKGROUND',     (0, 0), (-1, 0), colors.HexColor('#4361ee')),
        ('TEXTCOLOR',      (0, 0), (-1, 0), colors.white),
        ('FONTNAME',       (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('GRID',           (0, 0), (-1, -1), 0.5, colors.HexColor('#e0e0e0')),
        ('FONTSIZE',       (0, 0), (-1, -1), 10),
        ('PADDING',        (0, 0), (-1, -1), 8),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1),
         [colors.white, colors.HexColor('#f0f4ff')]),
        ('BACKGROUND',     (0, 1), (-1, 1), colors.HexColor('#edfbf0')),
    ]))
    story.append(top5_table)
    story.append(Spacer(1, 15))
    
    # ── SKIN IMAGE ──
    story.append(Paragraph("📸 Analyzed Image", heading_style))
    if os.path.exists(str(image_path)):
        img = Image(str(image_path), width=2.5*inch, height=2.5*inch)
        story.append(img)
    story.append(Spacer(1, 15))
    
    # ── RECOMMENDATIONS ──
    story.append(Paragraph("💡 Recommendations", heading_style))
    recommendation = prediction_result.get("recommendation", 
                                           "Consult a healthcare professional.")
    story.append(Paragraph(f"• {recommendation}", normal_style))
    story.append(Spacer(1, 10))
    
    # ── DISCLAIMER ──
    story.append(HRFlowable(width="100%", thickness=1, 
                             color=colors.HexColor('#e0e0e0')))
    story.append(Spacer(1, 10))
    
    disclaimer_style = ParagraphStyle(
        'Disclaimer',
        parent=styles['Normal'],
        fontSize=8,
        textColor=colors.HexColor('#999999'),
        alignment=TA_CENTER
    )
    story.append(Paragraph(
        "⚠️ DISCLAIMER: This report is generated by an AI model for "
        "educational purposes only. It is NOT a medical diagnosis. "
        "Always consult a qualified doctor or veterinarian for proper "
        "medical advice and treatment.",
        disclaimer_style
    ))
    
    # Build PDF
    doc.build(story)
    print(f"✅ PDF report saved to {output_path}")

print("✅ PDF generator function defined!")

✅ PDF generator function defined!


In [3]:
# Cell 3 - Test PDF generation
# Sample patient data
patient_data = {
    "name":      "Test Patient",
    "age":       35,
    "gender":    "Male",
    "species":   "Human",
    "duration":  "2-4 weeks",
    "itchy":     True,
    "spreading": True,
    "allergies": "None"
}

# Sample prediction result
prediction_result = {
    "disease":        "Warts Molluscum",
    "confidence":     82.91,
    "severity":       "Severe",
    "recommendation": "Immediate dermatologist consultation required.",
    "top5": [
        ("human_warts_molluscum",       82.91),
        ("human_atopic_dermatitis",     16.20),
        ("human_basal_cell_carcinoma",   0.50),
        ("human_tinea_ringworm",         0.30),
        ("human_psoriasis",              0.10),
    ]
}

# Output path
output_path = BASE_DIR / "reports" / "sample_report.pdf"
image_path  = BASE_DIR / "test_image.jpg"

# Generate PDF
generate_pdf_report(
    patient_data,
    prediction_result,
    image_path,
    output_path
)

print(f"\n🎉 PDF generated successfully!")
print(f"📄 Open it here: {output_path}")

✅ PDF report saved to C:\Users\shruti\Desktop\project\Skin-Disease Predictor\reports\sample_report.pdf

🎉 PDF generated successfully!
📄 Open it here: C:\Users\shruti\Desktop\project\Skin-Disease Predictor\reports\sample_report.pdf


In [4]:
# Cell 4 - Save pdf_report.py to utils folder
import shutil

# Copy notebook code to utils folder
pdf_code = '''from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, HRFlowable
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from pathlib import Path
from datetime import datetime
import os

def generate_pdf_report(patient_data, prediction_result, image_path, output_path):
    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=A4,
        rightMargin=50,
        leftMargin=50,
        topMargin=50,
        bottomMargin=50
    )
    styles = getSampleStyleSheet()
    story  = []

    title_style = ParagraphStyle(
        "Title",
        parent=styles["Title"],
        fontSize=24,
        textColor=colors.HexColor("#1a1a2e"),
        spaceAfter=5,
        alignment=TA_CENTER
    )
    subtitle_style = ParagraphStyle(
        "Subtitle",
        parent=styles["Normal"],
        fontSize=11,
        textColor=colors.HexColor("#555555"),
        spaceAfter=20,
        alignment=TA_CENTER
    )
    heading_style = ParagraphStyle(
        "Heading",
        parent=styles["Heading2"],
        fontSize=13,
        textColor=colors.HexColor("#4361ee"),
        spaceBefore=15,
        spaceAfter=8
    )
    normal_style = ParagraphStyle(
        "Normal",
        parent=styles["Normal"],
        fontSize=10,
        spaceAfter=5
    )

    story.append(Paragraph("Skin Disease Prediction Report", title_style))
    story.append(Paragraph("AI-Powered Skin Analysis | For Educational Purposes Only", subtitle_style))
    story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor("#4361ee")))
    story.append(Spacer(1, 15))

    story.append(Paragraph("Patient Information", heading_style))
    patient_table_data = [
        ["Field",     "Details"],
        ["Name",      patient_data.get("name",     "N/A")],
        ["Age",       str(patient_data.get("age",  "N/A"))],
        ["Gender",    patient_data.get("gender",   "N/A")],
        ["Species",   patient_data.get("species",  "Human")],
        ["Duration",  patient_data.get("duration", "N/A")],
        ["Itchy",     "Yes" if patient_data.get("itchy")     else "No"],
        ["Spreading", "Yes" if patient_data.get("spreading") else "No"],
        ["Allergies", patient_data.get("allergies", "None")],
        ["Date",      datetime.now().strftime("%d-%m-%Y %H:%M")]
    ]
    patient_table = Table(patient_table_data, colWidths=[2*inch, 4*inch])
    patient_table.setStyle(TableStyle([
        ("BACKGROUND",     (0, 0), (-1, 0), colors.HexColor("#4361ee")),
        ("TEXTCOLOR",      (0, 0), (-1, 0), colors.white),
        ("FONTNAME",       (0, 0), (-1, 0), "Helvetica-Bold"),
        ("GRID",           (0, 0), (-1, -1), 0.5, colors.HexColor("#e0e0e0")),
        ("FONTSIZE",       (0, 0), (-1, -1), 10),
        ("PADDING",        (0, 0), (-1, -1), 8),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f0f4ff")]),
    ]))
    story.append(patient_table)
    story.append(Spacer(1, 15))

    story.append(Paragraph("Prediction Results", heading_style))
    results_data = [
        ["Metric",           "Result"],
        ["Detected Disease", prediction_result.get("disease",    "N/A")],
        ["Confidence",       f"{prediction_result.get('confidence', 0)}%"],
        ["Severity Level",   prediction_result.get("severity",   "N/A")],
    ]
    results_table = Table(results_data, colWidths=[2*inch, 4*inch])
    results_table.setStyle(TableStyle([
        ("BACKGROUND",     (0, 0), (-1, 0), colors.HexColor("#4361ee")),
        ("TEXTCOLOR",      (0, 0), (-1, 0), colors.white),
        ("FONTNAME",       (0, 0), (-1, 0), "Helvetica-Bold"),
        ("GRID",           (0, 0), (-1, -1), 0.5, colors.HexColor("#e0e0e0")),
        ("FONTSIZE",       (0, 0), (-1, -1), 10),
        ("PADDING",        (0, 0), (-1, -1), 8),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f0f4ff")]),
    ]))
    story.append(results_table)
    story.append(Spacer(1, 15))

    story.append(Paragraph("Top 5 Predictions", heading_style))
    top5_data = [["Disease", "Confidence %"]]
    for disease, prob in prediction_result.get("top5", []):
        top5_data.append([disease.replace("_", " ").title(), f"{prob:.1f}%"])
    top5_table = Table(top5_data, colWidths=[4*inch, 2*inch])
    top5_table.setStyle(TableStyle([
        ("BACKGROUND",     (0, 0), (-1, 0), colors.HexColor("#4361ee")),
        ("TEXTCOLOR",      (0, 0), (-1, 0), colors.white),
        ("FONTNAME",       (0, 0), (-1, 0), "Helvetica-Bold"),
        ("GRID",           (0, 0), (-1, -1), 0.5, colors.HexColor("#e0e0e0")),
        ("FONTSIZE",       (0, 0), (-1, -1), 10),
        ("PADDING",        (0, 0), (-1, -1), 8),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f0f4ff")]),
    ]))
    story.append(top5_table)
    story.append(Spacer(1, 15))

    if os.path.exists(str(image_path)):
        story.append(Paragraph("Analyzed Image", heading_style))
        img = Image(str(image_path), width=2.5*inch, height=2.5*inch)
        story.append(img)
        story.append(Spacer(1, 15))

    story.append(Paragraph("Recommendations", heading_style))
    story.append(Paragraph(
        f"• {prediction_result.get('recommendation', 'Consult a healthcare professional.')}",
        normal_style
    ))
    story.append(Spacer(1, 10))

    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#e0e0e0")))
    story.append(Spacer(1, 10))
    disclaimer_style = ParagraphStyle(
        "Disclaimer",
        parent=styles["Normal"],
        fontSize=8,
        textColor=colors.HexColor("#999999"),
        alignment=TA_CENTER
    )
    story.append(Paragraph(
        "DISCLAIMER: This report is generated by an AI model for educational purposes only. "
        "It is NOT a medical diagnosis. Always consult a qualified doctor or veterinarian.",
        disclaimer_style
    ))
    doc.build(story)
'''

save_path = BASE_DIR / "utils" / "pdf_report.py"
with open(save_path, "w") as f:
    f.write(pdf_code)

print(f"✅ pdf_report.py saved to {save_path}")
print("🎉 PDF Report Notebook Complete!")

✅ pdf_report.py saved to C:\Users\shruti\Desktop\project\Skin-Disease Predictor\utils\pdf_report.py
🎉 PDF Report Notebook Complete!
